<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5509-notebooks/main/_banners/opim5509_banner.svg" width="100%" alt="OPIM 5509 banner"/>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5509-notebooks/blob/main/Module4/Motivating_ConvLSTM_and_Bidirectional.ipynb)

# Motivating ConvLSTM and Bidirectional
--------------------------------------------------
**Dr. Dave Wanik - University of Connecticut**

Two layers show up in Module 4.3 that we have not justified yet. The demand bake-off in `E3_Advanced_Demand_RNN`
is what makes the case, so start from its scoreboard:

| model | test MAE (MW) |
| :-- | --: |
| **Bidirectional LSTM** | **39.6** |
| plain LSTM | 50.7 |
| ConvLSTM | 52.4 |
| stacked LSTM | 53.5 |
| LSTM + recurrent dropout | 103.6 |
| persistence (last hour) | 129.0 |
| seasonal naive (same hour yesterday) | 227.1 |

Two results need explaining, and they pull in opposite directions:

1. **Bidirectional is the best model by 22%** - and the moment a student sees the word *backward* in a forecasting
   problem, they should ask whether we are cheating. **We are not**, and Part 5 walks the indices to show why.
2. **ConvLSTM is slightly WORSE than the plain LSTM here.** So the motivation cannot be "it wins." It is about
   what convolution buys you - and on this particular series, it turns out not to be worth paying for.

Nothing is fitted in this notebook. Every count is checked against Keras.

## The whole idea, before any arithmetic

Both layers sit **in front of, or around, the recurrent layer** you already understand.

- **Conv1D** slides a small window along the sequence and reports what it finds. It is a **learned smoother**:
  it turns raw hourly readings into a shorter sequence of local *shapes*.
- **Bidirectional** runs two independent copies of the recurrent layer - one reading oldest-to-newest, one
  newest-to-oldest - and **glues their answers together**.

Neither changes the data prep. The samples are the same `(samples, look-back, features)` tensors from Module 4.1.

In [1]:
import numpy as np
import pandas as pd
import keras
from keras import Input
from keras.models import Sequential
from keras.layers import Conv1D, MaxPooling1D, LSTM, Bidirectional, Dense
keras.utils.set_random_seed(5509)

def check(model, **by_hand):
    """Compare a hand count to Keras layer by layer, and raise if any line disagrees."""
    rows = []
    for (label, hand), layer in zip(by_hand.items(), model.layers):
        rows.append({"layer": layer.name, "by hand": hand, "Keras": layer.count_params(),
                     "output shape": str(layer.output.shape)})
    rows.append({"layer": "TOTAL", "by hand": sum(by_hand.values()),
                 "Keras": model.count_params(), "output shape": ""})
    out = pd.DataFrame(rows)
    out["match"] = out["by hand"] == out["Keras"]
    assert out["match"].all(), "the hand count disagrees with Keras:\n" + out.to_string()
    return out

🔴
<!-- 🎙 DAVE TALKING POINTS — M4 · M4.3-a — Conv1D on a sequence: a learned smoother
- Start on the E3 scoreboard - bidirectional wins by 22%, ConvLSTM LOSES to the plain LSTM. Both need explaining.
- Conv1D is NOT Conv2D. One filter is a small patch that slides along TIME, and it sees every feature at once.
- kernel_size=3 on 8 features = 24 weights + 1 bias per filter. 32 filters -> 800. Count the arrows, same as always.
- The length shrinks: 24 hours -> 22 steps, because a width-3 window only fits 22 times. Then pooling halves it.
- Say what it BUYS: a shorter, smoother sequence of local shapes, and an LSTM that spins 11 times instead of 24.
- Then say what it COSTS here: resolution. On a clean hourly load curve there is little noise to remove, which is exactly why ConvLSTM came 3rd in the bake-off. It would pay on something spikier.
-->


## Part 1 · What `Conv1D` does to a sequence

One sample is still a 24 x 8 block: 24 hours down, 8 features across. A `Conv1D` **filter** is a small patch that
slides **along time** and looks at **every feature at once**.

With `kernel_size=3` on 8 features, one filter holds

$$
\underbrace{3}_{\text{hours}} \times \underbrace{8}_{\text{features}} = 24 \text{ weights}, \quad +1 \text{ bias}
$$

and 32 filters cost $32 \times (24 + 1) = \mathbf{800}$. Counting arrows, exactly as before.

**The length shrinks.** A width-3 window fits into 24 hours only 22 times, so `(24, 8)` becomes `(22, 32)`:
22 time steps, each described by 32 numbers instead of 8.

$$
\text{output length} = \text{look-back} - \text{kernel\_size} + 1 = 24 - 3 + 1 = 22
$$

In [2]:
m = Sequential()
m.add(Input(shape=(24, 8)))                 # one sample: 24 hours x 8 features
m.add(Conv1D(filters=32, kernel_size=3, activation="relu"))
m.summary()

check(m, conv1d=32 * (3 * 8 + 1))

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 22, 32)         │           800 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 800 (3.12 KB)

 Trainable params: 800 (3.12 KB)

 Non-trainable params: 0 (0.00 B)

,layer,by hand,Keras,output shape,match
0,conv1d,800,800,"(None, 22, 32)",True
1,TOTAL,800,800,,True


### Why "a learned smoother"

Each filter answers one question about a 3-hour stretch - *is demand ramping? is it flat? did temperature jump
while demand did not?* - and it asks that same question at every position. The layer does not know what "morning"
means; it learns a handful of local shapes and reports where they occur.

## Part 2 · `MaxPooling1D` costs nothing and halves the length

Pooling has **no trainable parameters at all** - it is pure arithmetic. `pool_size=2` keeps the larger of each
adjacent pair, so `(22, 32)` becomes `(11, 32)`.

$$
22 \div 2 = 11 \text{ steps}, \qquad \text{parameters} = \mathbf{0}
$$

The point is not the saving. It is that the recurrent layer now spins **11 times instead of 24** and sees a
sequence of *summaries* rather than raw hours.

In [3]:
m = Sequential()
m.add(Input(shape=(24, 8)))
m.add(Conv1D(filters=32, kernel_size=3, activation="relu"))
m.add(MaxPooling1D(pool_size=2))
m.summary()

check(m, conv1d=32 * (3 * 8 + 1), max_pooling1d=0)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_1 (Conv1D)               │ (None, 22, 32)         │           800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 11, 32)         │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 800 (3.12 KB)

 Trainable params: 800 (3.12 KB)

 Non-trainable params: 0 (0.00 B)

,layer,by hand,Keras,output shape,match
0,conv1d_1,800,800,"(None, 22, 32)",True
1,max_pooling1d,0,0,"(None, 11, 32)",True
2,TOTAL,800,800,,True


## Part 3 · The ConvLSTM chain, end to end

Now put the LSTM behind it. The LSTM's $i$ is **32** - the filters, not the original 8 features - because that is
what arrives at each of the 11 steps.

$$
\begin{aligned}
\textbf{Conv1D}(32, k{=}3) & = 32\,(3 \cdot 8 + 1) = \mathbf{800} \\
\textbf{MaxPooling1D}(2) & = \mathbf{0} \\
\textbf{LSTM}(32) & = 4\,[\,32(32+32) + 32\,] = 4\,[\,2{,}048 + 32\,] = \mathbf{8{,}320} \\
\textbf{Dense}(1) & = 32 \cdot 1 + 1 = \mathbf{33} \\
\textbf{Total} & = \mathbf{9{,}153}
\end{aligned}
$$

Follow the shapes: `(24, 8) → (22, 32) → (11, 32) → (32,) → (1,)`.

In [4]:
convlstm = Sequential()
convlstm.add(Input(shape=(24, 8)))
convlstm.add(Conv1D(filters=32, kernel_size=3, activation="relu"))
convlstm.add(MaxPooling1D(pool_size=2))
convlstm.add(LSTM(32))
convlstm.add(Dense(1))
convlstm.summary()

check(convlstm, conv1d=800, max_pooling1d=0, lstm=4 * (32 * (32 + 32) + 32), dense=33)

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_2 (Conv1D)               │ (None, 22, 32)         │           800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 11, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 32)             │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,153 (35.75 KB)

 Trainable params: 9,153 (35.75 KB)

 Non-trainable params: 0 (0.00 B)

,layer,by hand,Keras,output shape,match
0,conv1d_2,800,800,"(None, 22, 32)",True
1,max_pooling1d_1,0,0,"(None, 11, 32)",True
2,lstm,8320,8320,"(None, 32)",True
3,dense,33,33,"(None, 1)",True
4,TOTAL,9153,9153,,True


**And yet it came third.** 52.4 MW against the plain LSTM's 50.7. Hourly load is already smooth - there is
very little high-frequency noise for a convolution to remove, and pooling throws away resolution the LSTM was
happily using. Conv1D earns its keep on **long, noisy, spiky** sequences; a clean 24-hour load curve is neither.

🔴
<!-- 🎙 DAVE TALKING POINTS — M4 · M4.3-b — Bidirectional: two cells, glued - and no, it is not cheating
- Bidirectional won the bake-off by 22%. Somebody WILL ask if reading backwards means seeing the future.
- First the mechanics: two independent LSTMs, one forward, one backward, outputs CONCATENATED. Width doubles, parameters double. Bidirectional(LSTM(32)) on 8 features = 2 x 5,248 = 10,496.
- Then the question. Walk the indices on screen: X = rows i..i+23, y = row i+24. EVERY row of the window is before the target. Reading them backwards reads the SAME rows in reverse - nothing new enters.
- So the rule is not about direction. It is about WHAT IS IN THE WINDOW. Leakage comes from centred windows (t-k..t+k predicting t), which is normal in sequence labelling and NLP and wrong for forecasting.
- Close on the real costs: 2x the parameters, and you cannot emit a prediction until the window is complete. That is latency, not leakage - and for day-ahead forecasting nobody cares.
-->


## Part 4 · What `Bidirectional` actually does

It makes **two independent copies** of the layer you wrap. One reads the sample oldest → newest, the other newest
→ oldest. Their final hidden states are **concatenated**.

So `Bidirectional(LSTM(32))` produces **64** numbers, not 32, and costs exactly twice a plain `LSTM(32)`:

$$
\textbf{Bidirectional}(\textbf{LSTM}(32)) = 2 \times 4\,[\,32(32+8) + 32\,] = 2 \times 5{,}248 = \mathbf{10{,}496}
$$

The `Dense` head then sees 64 inputs instead of 32.

In [5]:
bi = Sequential()
bi.add(Input(shape=(24, 8)))
bi.add(Bidirectional(LSTM(32)))
bi.add(Dense(1))
bi.summary()

one_way = 4 * (32 * (32 + 8) + 32)
print(f"a plain LSTM(32) on 8 features: {one_way:,}")
print(f"bidirectional is exactly twice:  {2 * one_way:,}")
check(bi, bidirectional=2 * one_way, dense=64 * 1 + 1)

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 64)             │        10,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,561 (41.25 KB)

 Trainable params: 10,561 (41.25 KB)

 Non-trainable params: 0 (0.00 B)

a plain LSTM(32) on 8 features: 5,248
bidirectional is exactly twice:  10,496


,layer,by hand,Keras,output shape,match
0,bidirectional,10496,10496,"(None, 64)",True
1,dense_1,65,65,"(None, 1)",True
2,TOTAL,10561,10561,,True


## Part 5 · "Isn't reading backwards seeing the future?"

This is the right question to ask, and the answer here is **no**. It is worth being precise, because the sloppy
version of this rule - *"never use bidirectional for forecasting"* - is wrong, and it would have you throw away
the best model in the bake-off.

### Walk the indices

This is how the energy notebook builds a sample:

```python
X.append(seqs[i : i + n_steps, :])     # rows i ... i+23
y.append(seqs[i + n_steps, -1])        # row i+24
```

So for one sample: the model reads **hours i through i+23** and predicts demand at **hour i+24**.

| | timestamps read | target |
| :-- | :-- | :-- |
| forward pass | i → i+23 | i+24 |
| **backward pass** | i+23 → i | i+24 |

**Both passes read exactly the same 24 rows.** The backward one visits them in a different order. Every row is
strictly before the target either way, so **nothing from after the prediction point ever enters the model**.

### So what WOULD be cheating?

Leakage is not about direction - it is about **what is in the window**. Build a *centred* window and you leak
regardless of which way you read it:

```python
X.append(seqs[t - k : t + k + 1])      # <-- rows AFTER t are in the input
y.append(seqs[t, -1])                  #     predicting t
```

Now rows $t{+}1 \dots t{+}k$ are inputs, and even a plain forward LSTM is cheating. That shape is completely
normal in **sequence labelling** and in **NLP** - tagging a word using the whole sentence, where later words are
genuinely available - and it is wrong for forecasting, where they are not.

> **The rule:** a model may read its window in any direction. It may not have anything in that window that it
> would not possess at prediction time.

### The real costs of bidirectional

Not leakage. Two practical things:

1. **Double the parameters** - 10,496 against 5,248 - for a model that already beats the baselines.
2. **You must wait for the whole window.** The backward pass cannot start until the last hour of the window has
   arrived, so there is no partial, streaming prediction. For day-ahead demand forecasting nobody cares; for a
   real-time controller it might matter.

That is the honest summary: **bidirectional is legitimate here, and it won.**

## What to remember

- **Conv1D counts like everything else:** `filters × (kernel_size × features + 1)`. It shortens the sequence to
  `look-back − kernel_size + 1`, and pooling halves it again for free.
- **Conv1D is a learned smoother.** It pays on noisy, spiky, long sequences. Hourly electricity load is smooth, so
  here it slightly *hurt* - 52.4 MW against the plain LSTM's 50.7.
- **Bidirectional is two cells concatenated:** double the width, double the parameters.
- **Backward is not cheating when every row of the window precedes the target.** Leakage comes from *centred
  windows*, not from direction. Bidirectional won this bake-off fairly, at 39.6 MW.

Next: **`E3_Advanced_Demand_RNN`** runs all five architectures on the real demand series and scores them.

## Terminology

The terms used in this module, and where each one appears in the figures.

| Term | Symbol / code | In the figures | Definition |
| :-- | :-- | :-- | :-- |
| **look-back** | `n_steps` | the green window sliding down the series | the number of time steps in one sample |
| **features** | $i$, `n_features` | the **green dots** | the number of columns supplied at each time step |
| **hidden units** | $h$, the number in `SimpleRNN(h)` | the **red dots** | the width of the layer's memory; chosen by you |
| **hidden state** | $h_t$ | one set of red dots | the layer's memory after reading step $t$ |
| **sequence of hidden states** | $h_1 \dots h_T$ | all the red dots, left to right | one hidden state per time step; what the layer produces |
| **`return_sequences`** | `True` / `False` | - | return all hidden states, or only the final one |
| **$g$** | - | the small networks drawn inside one cell | SimpleRNN 1 · GRU 3 · LSTM 4 |

**Shapes.** A batch of samples has shape `(samples, look-back, features)`. A recurrent layer maps each sample to
`(look-back, hidden units)` when `return_sequences=True`, and to `(hidden units,)` otherwise.

**Parameters.** $g\,[\,h(h+i) + h\,]$ for a recurrent layer and $n_{in}n_{out} + n_{out}$ for a dense layer.
Neither expression contains the look-back.

### Further reading

**Chollet, *Deep Learning with Python* (2nd ed.), Chapter 10, "Deep learning for timeseries"** - §10.2 for
recurrent layers and the hidden state, §10.3 for stacking, recurrent dropout and bidirectional layers. The full
text is available at no cost through **library.uconn.edu** with your NetID.

The cell animations in these notebooks are **Raimi Karim's**. Note that the original `towardsdatascience.com` links
are no longer valid - Towards Data Science moved off Medium and those addresses now resolve elsewhere. The current
addresses are:

* **Animated RNN, LSTM and GRU** (Dec 2018) - https://medium.com/data-science/animated-rnn-lstm-and-gru-ef124d06cf45
* **Counting No. of Parameters in Deep Learning Models by Hand** (Jan 2019) - https://medium.com/data-science/counting-no-of-parameters-in-deep-learning-models-by-hand-8f1716241889